# Lab 1b: When Pandas Breaks

**Week 1 — Foundations | Big Data Engineering Course**

## Objective
Experience firsthand why distributed computing exists. We'll load progressively larger
subsets of the NYC Yellow Taxi dataset until pandas can no longer handle it, then
measure memory usage and execution time.

## Dataset
**NYC Yellow Taxi Trip Records** — ~3 million rows per monthly Parquet file

## Prerequisites
- Run `python setup/check_environment.py` to verify your setup
- Run `python setup/download_datasets.py --mode sample` to get the data

In [ ]:
import pandas as pd
import numpy as np
import time
import os
import psutil
from pathlib import Path

# Course data directory
DATA_DIR = Path("../../data/sample/taxi")
print(f"Data directory: {DATA_DIR}")
print(f"Files: {list(DATA_DIR.glob('*.parquet'))}")

## Part 1: Profiling Pandas Performance

Let's build a utility to measure how pandas performs as data grows.

In [ ]:
def measure_pandas_performance(filepath, n_rows=None):
    """Load data with pandas and measure time + memory."""
    process = psutil.Process(os.getpid())
    mem_before = process.memory_info().rss / 1024**2  # MB
    
    start = time.time()
    if n_rows:
        df = pd.read_parquet(filepath).head(n_rows)
    else:
        df = pd.read_parquet(filepath)
    load_time = time.time() - start
    
    mem_after = process.memory_info().rss / 1024**2
    
    # Run a typical aggregation
    start = time.time()
    result = (
        df.groupby(df['tpep_pickup_datetime'].dt.hour)
        .agg({
            'total_amount': ['mean', 'sum', 'count'],
            'trip_distance': 'mean',
            'passenger_count': 'sum'
        })
    )
    agg_time = time.time() - start
    
    return {
        'rows': len(df),
        'columns': len(df.columns),
        'memory_mb': round(mem_after - mem_before, 1),
        'load_time_s': round(load_time, 2),
        'agg_time_s': round(agg_time, 2),
        'df_memory_mb': round(df.memory_usage(deep=True).sum() / 1024**2, 1),
    }

print("Profiler ready.")

## Part 2: Scaling Up

Load increasingly large portions of the taxi data and observe how pandas copes.

In [ ]:
# Find available parquet files
parquet_files = sorted(DATA_DIR.glob('*.parquet'))
if not parquet_files:
    print("No data found! Run: python setup/download_datasets.py --mode sample")
else:
    filepath = parquet_files[0]
    print(f"Using: {filepath}")
    print(f"File size: {filepath.stat().st_size / 1024**2:.1f} MB")

    # Test with progressively larger subsets
    row_counts = [10_000, 100_000, 500_000, 1_000_000, None]  # None = full file
    results = []
    
    for n in row_counts:
        label = f"{n:,}" if n else "ALL"
        print(f"\nTesting with {label} rows...")
        try:
            r = measure_pandas_performance(filepath, n)
            r['label'] = label
            results.append(r)
            print(f"  Rows: {r['rows']:,} | Memory: {r['df_memory_mb']} MB | "
                  f"Load: {r['load_time_s']}s | Agg: {r['agg_time_s']}s")
        except MemoryError:
            print(f"  MEMORY ERROR at {label} rows!")
            break

In [ ]:
# Visualize the scaling behavior
import matplotlib.pyplot as plt

if results:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    rows = [r['rows'] for r in results]
    
    axes[0].plot(rows, [r['df_memory_mb'] for r in results], 'o-', color='#e74c3c')
    axes[0].set_xlabel('Number of Rows')
    axes[0].set_ylabel('DataFrame Memory (MB)')
    axes[0].set_title('Memory Usage')
    
    axes[1].plot(rows, [r['load_time_s'] for r in results], 's-', color='#3498db')
    axes[1].set_xlabel('Number of Rows')
    axes[1].set_ylabel('Load Time (seconds)')
    axes[1].set_title('Load Time')
    
    axes[2].plot(rows, [r['agg_time_s'] for r in results], '^-', color='#2ecc71')
    axes[2].set_xlabel('Number of Rows')
    axes[2].set_ylabel('Aggregation Time (seconds)')
    axes[2].set_title('Aggregation Time')
    
    for ax in axes:
        ax.ticklabel_format(style='plain', axis='x')
    
    plt.tight_layout()
    plt.suptitle('Pandas Performance Scaling — NYC Taxi Data', y=1.02, fontsize=14)
    plt.show()

## Part 3: The Breaking Point

Now let's try to simulate what happens with a full year of taxi data (~36M rows).
We'll duplicate the data in memory to stress-test pandas.

In [ ]:
# How much RAM do we have?
total_ram = psutil.virtual_memory().total / 1024**3
available_ram = psutil.virtual_memory().available / 1024**3
print(f"Total RAM: {total_ram:.1f} GB")
print(f"Available RAM: {available_ram:.1f} GB")

if results:
    full_file_memory = results[-1]['df_memory_mb']
    year_estimate = full_file_memory * 12  # 12 months
    print(f"\nSingle month in memory: {full_file_memory:.0f} MB")
    print(f"Estimated full year: {year_estimate:.0f} MB ({year_estimate/1024:.1f} GB)")
    print(f"Available RAM: {available_ram*1024:.0f} MB")
    
    if year_estimate > available_ram * 1024:
        print("\n** A full year would EXCEED available RAM! **")
        print("This is exactly why we need Spark and Dask.")
    else:
        print("\nA full year might fit in RAM, but processing it efficiently")
        print("(joins, aggregations, ML) would still benefit from distributed computing.")

## Key Takeaways

1. **Pandas loads everything into memory** — the DataFrame uses 2-10x the file size in RAM
2. **Performance degrades linearly** (at best) as data grows
3. **Memory is the hard ceiling** — you simply can't load data larger than RAM
4. **Even within RAM, operations slow down** — groupby/join on 100M+ rows is painful

## What's Next

In the next notebook (`01c_first_spark_session.ipynb`), we'll load this same taxi data
using PySpark and see how lazy evaluation and distributed processing change the game.

---
*Big Data Engineering: Spark, Dask & AWS — Week 1, Lab 1b*